# simulation

Every simulation of the shape-matching experiment, in one notebook. It replaces the Python scripts in `src/`
(adviser, 2026-10-26). Balabit only.

**How to use**

1. Open this notebook with the **`.venv` kernel** (it has numba). VS Code: *Select Kernel* → `.venv`; Jupyter:
   *Kernel → Change kernel*.
2. Edit the **Settings** cell below. Nothing else needs changing.
3. **Run All.** Every section prints what it wrote. Sections switched off in `SECTIONS` are skipped.

**Outputs are CSV files only**, under `experiment_earl/runs/<hash>/`. The 6-character `<hash>` is made from
`SETTINGS`, so changing a setting writes to a new folder and never overwrites an earlier result. `runs/` and `cache/`
are git-ignored.

| Section | Writes |
|---|---|
| 3. Batch | `runs/<hash>/<L>_<I>_<sensitivity>_<sessions>/` per run: `config.yaml`, `run.json`, `results/accuracy.csv`, `results/session_progression*.csv`; `runs/batch_log.csv` |
| 4. Accuracy | `runs/<hash>/accuracy.csv` (every run and user), `runs/<hash>/summary/summary.csv` (one row per run) |
| 5. Best per split | `runs/<hash>/best/`: `scores.csv`, `best.csv`, `best_rows.csv` |
| 6. Sensitivity sweep | `runs/<hash>/sweep/`: `confusion.csv`, `user_confusion.csv`, `parts/`, `config.yaml` |
| 7. Best setting per user | `runs/<hash>/sweep/best/`: `best_users.csv`, `best_splits.csv` |
| 8. Matched pairs | `results/matches.csv` of `SINGLE_RUN` |
| 9. Genuine check | `results/genuine_check.csv`, `results/genuine_check_summary.csv` of `SINGLE_RUN` |

The full default batch (9 splits × 50 sensitivities × 15 session counts = 6,750 runs) takes many hours. Shrink
`BATCH` to try the notebook out first.

In [1]:
import hashlib
import json
import pickle
import random
import shutil
import subprocess
import time
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

try:
    from numba import njit, prange
except ImportError:
    raise ImportError("numba is not installed in this kernel: choose the .venv kernel, "
                      "or install it here with  %pip install numba") from None

_HERE = Path.cwd()
ROOT = next((p for p in [_HERE, _HERE / "experiment_earl", *_HERE.parents] if (p / "datasets" / "balabit").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError(f"No datasets/balabit found from {_HERE}: open this notebook from inside the repository")
RUNS = ROOT / "runs"        # every result, one folder per settings hash
CACHE_DIR = ROOT / "cache"  # DTW results reused across runs (keys hold the settings that change a score)
print("experiment folder:", ROOT)

experiment folder: /home/ejang/Desktop/Thesis/thesis-mouse-biometrics/experiment_earl


## Settings

Everything the simulation uses. `SETTINGS` names the results folder `runs/<hash>/`; the other blocks say what to run.

In [2]:
SETTINGS = {
    "seed": 900,                # picks the users and each user's session order once (same seed = same draw)
    "dataset_dir": "datasets/balabit/users",              # Balabit, one folder per user (relative to experiment_earl/)
    "labels_file": "datasets/balabit/public_labels.csv",  # sessions with is_illegal = 1 are dropped; None keeps all
    # chunking: cut a session into strokes
    "pause_gap_s": 0.5,         # a pause longer than this many seconds starts a new stroke (clicks also split strokes)
    "min_points": 10,           # strokes with fewer recorded points are dropped as jitter
    "min_length_px": 0,         # strokes with a shorter path (px) are dropped (0 = no filter)
    "resample_n": 64,           # every stroke is redrawn with this many evenly spaced points before matching
    # matching
    "dtw_band": 0.1,            # DTW may stretch a stroke by at most this share of resample_n
    "match_shape_ratio": 0.2,   # a session passes when at least this share of its strokes match
    "match_session_ratio": 0.5, # a user goes through when at least this share of their sessions pass
}

# 3. Batch: every split x sensitivity x sessions is one run, in runs/<hash>/<L>_<I>_<sensitivity>_<sessions>/
BATCH = {
    "splits": [[1, 9]],  # [legitimate, impostor] users, at most 10 together
    "sensitivities": [round(0.01 * i, 2) for i in range(1, 5)],  # strokes match when the DTW distance per point is <= this: 0.01 ... 0.50
    "sessions": list(range(2, 31, 2)),                             # sessions per user: 2, 4, ... 30
    "session_progression": True,  # also count impostors matched per step 1..sessions (off = accuracy only, faster)
}
SKIP_FINISHED_RUNS = True  # a run whose results/accuracy.csv exists is not simulated again

# 6. Sensitivity sweep: per fixed split, one confusion matrix (positive = impostor) per library size and sensitivity
SWEEP = {
    "splits": [[1, 5], [5, 1], [3, 2], [2, 3]],
    "sensitivity": [0.01, 1.00, 0.01],  # start, stop, step of the sensitivities scored
    "sessions_max": 40,                 # library sizes 1, 2, ... up to this many of a legitimate user's first sessions
    "test_sessions": 40,                # tested per user: a legitimate user's last this many, an impostor's first this many
}

# 8-9. One run for the optional checks
SINGLE_RUN = {"n_legitimate": 3, "n_impostor": 7, "sensitivity": 0.02, "sessions_per_user": 10}
GENUINE_HELD_OUT = 2  # genuine check: the legitimate users' next sessions after sessions_per_user (never in a profile)

SECTIONS = {
    "batch": True,          # 3. simulate every run of BATCH
    "accuracy": True,       # 4. collect accuracy.csv and summary.csv of every finished run in runs/<hash>/
    "best": True,           # 5. best sensitivity and sessions per split (needs 4)
    "sweep": True,          # 6. sensitivity sweep
    "best_setting": True,   # 7. best setting per legitimate user (needs 6)
    "matches": False,       # 8. matched stroke pairs of SINGLE_RUN
    "genuine_check": False, # 9. own unused sessions of SINGLE_RUN against their own profile
}

## 1. Shared code

Run folders, the Balabit loader, strokes and banded DTW. Nothing to change here.

In [3]:
RUN_KEYS = ("n_legitimate", "n_impostor", "sensitivity", "sessions_per_user")  # name a run inside its hash folder
INVALID_COORDINATE = 65535  # Balabit logging glitch, not a position


def settings_hash(config: dict) -> str:
    """6-character hash of every setting except RUN_KEYS, so runs that differ only in those sit side by side."""
    settings = {k: v for k, v in config.items() if k not in RUN_KEYS}
    return hashlib.sha1(json.dumps(settings, sort_keys=True).encode()).hexdigest()[:6]


def group_folder() -> Path:
    """runs/<hash>/ of SETTINGS: the batch runs, accuracy, best and sweep of these settings."""
    return RUNS / settings_hash(SETTINGS)


def run_folder(config: dict) -> Path:
    name = "_".join(f"{config[k]:.2f}" if k == "sensitivity" else f"{config[k]:g}" for k in RUN_KEYS)
    return RUNS / settings_hash(config) / name


def create_run(folder: Path, config: dict) -> None:
    """Run folder with its settings (config.yaml) and run.json (creation time, git commit)."""
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "config.yaml").write_text(yaml.safe_dump(config, sort_keys=False))
    try:
        commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True,
                                cwd=ROOT, check=True).stdout.strip()
    except (OSError, subprocess.CalledProcessError):
        commit = None
    (folder / "run.json").write_text(json.dumps(
        {"created": datetime.now().isoformat(timespec="seconds"), "git_commit": commit}, indent=2) + "\n")


def write_csv(table: pd.DataFrame, path: Path, index: bool = False) -> None:
    """Write via a temporary file, so an interrupted write never leaves a half file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    table.to_csv(tmp, index=index)
    tmp.replace(path)


_ILLEGAL: dict = {}


def illegal_sessions(config: dict) -> set[str]:
    """Sessions that labels_file marks is_illegal = 1: recorded by someone else in this user's folder."""
    if not config.get("labels_file"):
        return set()
    if config["labels_file"] not in _ILLEGAL:
        labels = pd.read_csv(ROOT / config["labels_file"])
        _ILLEGAL[config["labels_file"]] = set(labels.loc[labels["is_illegal"] == 1, "filename"])
    return _ILLEGAL[config["labels_file"]]


def list_users(config: dict) -> list[str]:
    return sorted(p.name for p in (ROOT / config["dataset_dir"]).iterdir() if p.is_dir())


def list_sessions(config: dict, user: str) -> list[Path]:
    """Every session of `user` (Balabit's training and test files merged per user, adviser 2026-10-14)."""
    illegal = illegal_sessions(config)
    return sorted(p for p in (ROOT / config["dataset_dir"] / user).iterdir() if p.is_file() and p.name not in illegal)


def session_pool(config: dict, user: str, seed: int) -> list[str]:
    """Session names of `user`, shuffled once from `seed`, so step k+1 contains step k."""
    names = [p.name for p in list_sessions(config, user)]
    random.Random(f"{seed}-{user}").shuffle(names)
    return names


def max_sessions(config: dict) -> int:
    """Largest session count every user can supply."""
    return min(len(list_sessions(config, user)) for user in list_users(config))


def draw_users(config: dict, seed: int) -> tuple[list[str], list[str], dict[str, list[str]]]:
    """One draw from `seed`, fixed for the whole run: legitimate users in draw order, sorted impostors, and each
    user's shuffled session pool."""
    users = list_users(config)
    rng = random.Random(seed)
    legit = rng.sample(users, config["n_legitimate"])
    impostor = sorted(rng.sample([u for u in users if u not in legit], config["n_impostor"]))
    return legit, impostor, {user: session_pool(config, user, seed) for user in legit + impostor}


def load_session(path: Path) -> pd.DataFrame:
    """Columns t (seconds from the first event), x, y, button, state, in time order. Scroll rows and glitch rows
    (x or y = 65535) are dropped; the sort is stable because many Balabit timestamps tie."""
    df = pd.read_csv(path, usecols=["client timestamp", "button", "state", "x", "y"])
    df = df.rename(columns={"client timestamp": "t"}).dropna(subset=["t", "x", "y"])
    df = df[(df["button"] != "Scroll") & (df["x"] < INVALID_COORDINATE) & (df["y"] < INVALID_COORDINATE)]
    df = df.sort_values("t", kind="stable").reset_index(drop=True)
    df["t"] = df["t"] - df["t"].iloc[0] if len(df) else df["t"]
    return df[["t", "x", "y", "button", "state"]]

In [4]:
MOVE, DRAG = 0, 1
KIND_NAMES = {MOVE: "move", DRAG: "drag"}


def path_length(chunk: np.ndarray) -> float:
    return float(np.hypot(*np.diff(chunk[:, :2], axis=0).T).sum())


def segment(df: pd.DataFrame, config: dict) -> list[tuple[np.ndarray, int]]:
    """Strokes as ([x, y, t] array, kind). A new stroke starts after a pause or any click event."""
    click = df["state"].isin(["Pressed", "Released"]).to_numpy()
    clicks_before = np.cumsum(click)
    keep = ~click
    t = df["t"].to_numpy()[keep]
    if not len(t):
        return []
    xyt = np.column_stack([df["x"].to_numpy()[keep], df["y"].to_numpy()[keep], t])
    drag = df["state"].to_numpy()[keep] == "Drag"
    clicked = np.diff(clicks_before[keep], prepend=clicks_before[keep][0]) > 0
    new_chunk = (np.diff(t, prepend=t[0]) > config["pause_gap_s"]) | clicked
    starts = np.flatnonzero(new_chunk)
    starts = np.concatenate([[0], starts[starts > 0], [len(t)]])
    chunks = []
    for lo, hi in zip(starts[:-1], starts[1:]):
        if hi - lo >= config["min_points"] and path_length(xyt[lo:hi]) >= config["min_length_px"]:
            chunks.append((xyt[lo:hi], DRAG if drag[lo:hi].mean() > 0.5 else MOVE))
    return chunks


def normalize(chunk: np.ndarray, n: int) -> np.ndarray | None:
    """Start at the origin, bounding-box diagonal 1, n points evenly spaced by arc length."""
    xy = chunk[:, :2] - chunk[0, :2]
    diag = np.hypot(np.ptp(xy[:, 0]), np.ptp(xy[:, 1]))
    arc = np.concatenate([[0.0], np.cumsum(np.hypot(*np.diff(xy, axis=0).T))])
    if diag == 0 or arc[-1] == 0:
        return None
    grid = np.linspace(0, arc[-1], n)
    return np.column_stack([np.interp(grid, arc, xy[:, 0]), np.interp(grid, arc, xy[:, 1])]) / diag


@dataclass
class Session:
    key: str                      # "<user>/<session>"
    n_chunks: int                 # every recorded stroke, also ones too flat to compare
    paths: dict[int, np.ndarray] = field(default_factory=dict)  # kind -> (m, n, 2) normalized strokes


def segmentation(config: dict) -> tuple:
    """Settings that change which strokes a session has; part of every cache key."""
    return (config["pause_gap_s"], config["min_points"], config["min_length_px"], config["resample_n"])


_SESSIONS: dict = {}  # (session file, segmentation) -> Session, so each file is cut into strokes once


def dataset_session(user: str, name: str, config: dict) -> Session:
    path = ROOT / config["dataset_dir"] / user / name
    if (path, segmentation(config)) not in _SESSIONS:
        chunks = segment(load_session(path), config)
        session = Session(f"{user}/{name}", len(chunks))
        for kind in (MOVE, DRAG):
            paths = [normalize(c, config["resample_n"]) for c, k in chunks if k == kind]
            session.paths[kind] = np.array([p for p in paths if p is not None]).reshape(-1, config["resample_n"], 2)
        _SESSIONS[(path, segmentation(config))] = session
    return _SESSIONS[(path, segmentation(config))]


@njit(cache=True)
def _dtw_within(a, b, band, cap, prev, cur):
    """True if the banded DTW cost between paths a and b is at most `cap` (gives up once every cell exceeds it)."""
    n = a.shape[0]
    prev[:] = np.inf
    prev[0] = 0.0
    for i in range(n):
        cur[:] = np.inf
        row_min = np.inf
        for j in range(max(0, i - band), min(n - 1, i + band) + 1):
            dx, dy = a[i, 0] - b[j, 0], a[i, 1] - b[j, 1]
            value = np.sqrt(dx * dx + dy * dy) + min(prev[j], prev[j + 1], cur[j])
            cur[j + 1] = value
            row_min = min(row_min, value)
        if row_min > cap:
            return False
        prev, cur = cur, prev
    return prev[n] <= cap


@njit(parallel=True, cache=True)
def has_match(queries, library, band, cap):
    """Per query path: does any library path lie within DTW cost `cap`?"""
    n = queries.shape[1]
    found = np.zeros(queries.shape[0], dtype=np.bool_)
    for q in prange(queries.shape[0]):
        prev, cur = np.empty(n + 1), np.empty(n + 1)
        for k in range(library.shape[0]):
            if _dtw_within(queries[q], library[k], band, cap, prev, cur):
                found[q] = True
                break
    return found


PAIRS_PATH = CACHE_DIR / "pairs.pkl"
_PAIRS: dict = {}  # (query session, library session, segmentation, band, sensitivity) -> {kind: bool per query stroke}
if PAIRS_PATH.exists():
    _PAIRS.update(pickle.loads(PAIRS_PATH.read_bytes()))
_SAVED_PAIRS = set(_PAIRS)


def pair_matches(query: Session, library: Session, config: dict) -> dict[int, np.ndarray]:
    n = config["resample_n"]
    key = (query.key, library.key, segmentation(config), config["dtw_band"], config["sensitivity"])
    if key not in _PAIRS:
        band, cap = int(round(config["dtw_band"] * n)), config["sensitivity"] * n
        _PAIRS[key] = {kind: has_match(paths, library.paths[kind], band, cap) if len(library.paths[kind])
                       else np.zeros(len(paths), dtype=bool)
                       for kind, paths in query.paths.items()}
    return _PAIRS[key]


def matched_chunks(query: Session, library: list[Session], config: dict) -> int:
    """Strokes of `query` (same kind only) that match a stroke in any library session."""
    total = 0
    for kind, paths in query.paths.items():
        found = np.zeros(len(paths), dtype=bool)
        for session in library:
            found |= pair_matches(query, session, config)[kind]
        total += int(found.sum())
    return total


def forget_pairs(sensitivity: float) -> None:
    """Drop the pairs computed at `sensitivity` since the cache was loaded; no other sensitivity reuses them."""
    for key in [k for k in _PAIRS if k[-1] == sensitivity and k not in _SAVED_PAIRS]:
        del _PAIRS[key]


def save_pairs() -> None:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    tmp = PAIRS_PATH.with_suffix(".tmp")
    with tmp.open("wb") as handle:  # stream, so the cache is never held twice in memory
        pickle.dump(_PAIRS, handle)
    tmp.replace(PAIRS_PATH)
    _SAVED_PAIRS.update(_PAIRS)


print(f"{len(_PAIRS):,} cached DTW pairs loaded")

199,192 cached DTW pairs loaded


## 2. Data check

Users and sessions available, and whether the settings fit them.

In [5]:
users = list_users(SETTINGS)
fewest = max_sessions(SETTINGS)
total = sum(len(list_sessions(SETTINGS, u)) for u in users)
print(f"{len(users)} users, {total:,} sessions; every user has at least {fewest}")

problems = [f"split {split} needs {sum(split)} users; the data has {len(users)}"
            for split in BATCH["splits"] + SWEEP["splits"] + [[SINGLE_RUN["n_legitimate"], SINGLE_RUN["n_impostor"]]]
            if sum(split) > len(users)]
if 2 * max(BATCH["sessions"]) > fewest:
    problems.append(f"BATCH sessions up to {max(BATCH['sessions'])}: a legitimate user's last sessions would overlap "
                    f"the profile (at most {fewest // 2})")
if SWEEP["sessions_max"] + SWEEP["test_sessions"] > fewest:
    problems.append(f"SWEEP sessions_max + test_sessions = {SWEEP['sessions_max'] + SWEEP['test_sessions']} > {fewest}")
if SINGLE_RUN["sessions_per_user"] + GENUINE_HELD_OUT > fewest:
    problems.append(f"SINGLE_RUN sessions_per_user + GENUINE_HELD_OUT > {fewest}")
if problems:
    raise ValueError("Settings do not fit the data:\n- " + "\n- ".join(problems))
print("results folder:", group_folder())

10 users, 1,271 sessions; every user has at least 87
results folder: /home/ejang/Desktop/Thesis/thesis-mouse-biometrics/experiment_earl/runs/540c32


## 3. Batch: simulate every run

For each run (split, sensitivity, sessions per user), one draw from `seed` fixes the legitimate users, the impostors
and every user's session order.

- **Accuracy** (`results/accuracy.csv`, one row per user): a legitimate user's own **last** `sessions` sessions (never
  in a profile) against their own profile; an impostor's **first** `sessions` sessions against the legitimate profile
  that passes most of them. A session passes when at least `match_shape_ratio` of its strokes match; a user goes
  through when at least `match_session_ratio` of their sessions pass.
- **Session progression** (`results/session_progression*.csv`, if `BATCH["session_progression"]`): for 1, 2, ...
  `sessions` sessions per user and the first 1, 2, ... legitimate users, which impostors match a profile.

Runs are ordered by sensitivity, so DTW results are reused between runs of the same sensitivity.

In [6]:
def run_step(legit: list[str], impostor: list[str], order: dict, k: int, config: dict) -> list[dict]:
    """Each impostor's first k sessions against each legitimate profile (first k sessions); matched when any single
    profile passes at least match_session_ratio of them."""
    pick = lambda user: [dataset_session(user, name, config) for name in order[user][:k]]
    profiles = {user: pick(user) for user in legit}
    rows = []
    for user in impostor:
        sessions = pick(user)
        per_profile = {p: [matched_chunks(s, library, config) / max(s.n_chunks, 1) for s in sessions]
                       for p, library in profiles.items()}
        share = {p: float(np.mean([r >= config["match_shape_ratio"] for r in ratios])) for p, ratios in per_profile.items()}
        best = max(per_profile, key=lambda p: (share[p], np.mean(per_profile[p])))
        rows.append({
            "sessions": k, "impostor": user, "best_profile": best,
            "mean_chunk_ratio": round(float(np.mean(per_profile[best])), 3),
            "sessions_matched": f"{round(share[best] * len(sessions))}/{len(sessions)}",
            "matched": share[best] >= config["match_session_ratio"],
            "matched_by": ",".join(p for p, v in share.items() if v >= config["match_session_ratio"]),
        })
    return rows


def progression(config: dict, legit: list[str], impostor: list[str], order: dict) -> dict[str, pd.DataFrame]:
    """Every step 1..sessions_per_user with the first 1..n_legitimate legitimate users."""
    steps = pd.DataFrame([{"legit_users": n, "legitimate": ",".join(legit[:n]), **row}
                          for k in range(1, config["sessions_per_user"] + 1)
                          for n in range(1, len(legit) + 1)
                          for row in run_step(legit[:n], impostor, order, k, config)])
    per_step = (steps.groupby(["legit_users", "sessions"])["matched"]
                .agg(matched_detected="sum", sensitivity="mean").reset_index())
    summary = per_step[per_step["legit_users"] == len(legit)].set_index("sessions")[["matched_detected", "sensitivity"]]
    summary["n_legit"], summary["n_impostor"], summary["seed"] = len(legit), len(impostor), config["seed"]
    heatmap = per_step.pivot(index="sessions", columns="legit_users", values="sensitivity")
    full = steps[steps["legit_users"] == len(legit)]
    per_user = pd.DataFrame([{"profile": p, "sessions": r.sessions, "impostor": r.impostor,
                              "matched": p in r.matched_by.split(",")} for r in full.itertuples() for p in legit])
    per_user = per_user.groupby(["profile", "sessions"], sort=False)["matched"].sum().rename("impostors_matched").reset_index()
    return {"session_progression_steps": steps, "session_progression": summary.round(3),
            "session_progression_heatmap": heatmap.round(3), "session_progression_users": per_user}


def score(user: str, sessions: list[str], library: list[Session], config: dict) -> dict:
    """Sessions of `user` that pass (>= match_shape_ratio of their strokes match the library), plus the strokes."""
    passed = matched = strokes = 0
    for name in sessions:
        session = dataset_session(user, name, config)
        hits = matched_chunks(session, library, config)
        passed += hits / max(session.n_chunks, 1) >= config["match_shape_ratio"]
        matched += hits
        strokes += session.n_chunks
    return {"passed": passed, "tested": len(sessions), "matched_strokes": matched, "strokes": strokes}


def accuracy_rows(config: dict, name: str, legit: list[str], impostor: list[str], order: dict) -> list[dict]:
    """Legitimate: own last sessions_per_user sessions (never in a profile) against own profile. Impostor: first
    sessions_per_user sessions against the legitimate profile that passes most of them."""
    k = config["sessions_per_user"]
    libraries = {u: [dataset_session(u, n, config) for n in order[u][:k]] for u in legit}
    run = {"run": name, "sensitivity": config["sensitivity"], "sessions": k,
           "n_legitimate": config["n_legitimate"], "n_impostor": config["n_impostor"]}
    rows = [{**run, "kind": "legitimate", "user": u, "profile": u, **score(u, order[u][-k:], libraries[u], config)}
            for u in legit]
    for user in impostor:
        scores = {p: score(user, order[user][:k], lib, config) for p, lib in libraries.items()}
        profile = max(scores, key=lambda p: (scores[p]["passed"], scores[p]["matched_strokes"] / max(scores[p]["strokes"], 1)))
        rows.append({**run, "kind": "impostor", "user": user, "profile": profile, **scores[profile]})
    for row in rows:
        row["through"] = row["passed"] >= config["match_session_ratio"] * row["tested"]
    return rows


def run_done(folder: Path) -> bool:
    results = folder / "results"
    return (results / "accuracy.csv").exists() and (
        not BATCH["session_progression"] or (results / "session_progression.csv").exists())


def simulate(config: dict) -> None:
    """One run: its folder, then the session progression and accuracy CSVs (accuracy last: it marks the run done)."""
    folder = run_folder(config)
    if not (folder / "config.yaml").exists():
        create_run(folder, config)
    legit, impostor, order = draw_users(config, config["seed"])
    results = folder / "results"
    if BATCH["session_progression"]:
        for name, table in progression(config, legit, impostor, order).items():
            write_csv(table, results / f"{name}.csv", index=name in ("session_progression", "session_progression_heatmap"))
    write_csv(pd.DataFrame(accuracy_rows(config, folder.name, legit, impostor, order)), results / "accuracy.csv")


def log_run(row: dict) -> None:
    log = RUNS / "batch_log.csv"
    pd.DataFrame([row]).to_csv(log, mode="a", header=not log.exists(), index=False)


if SECTIONS["batch"]:
    plan = [{**SETTINGS, "n_legitimate": legit_n, "n_impostor": impostor_n, "sensitivity": sens, "sessions_per_user": k}
            for sens in BATCH["sensitivities"] for legit_n, impostor_n in BATCH["splits"] for k in BATCH["sessions"]]
    todo = [c for c in plan if not (SKIP_FINISHED_RUNS and run_done(run_folder(c)))]
    print(f"{len(plan):,} runs in the plan, {len(plan) - len(todo):,} finished earlier, {len(todo):,} to run")
    counts = {"ok": 0, "failed": 0}
    for i, config in enumerate(todo, 1):
        name = run_folder(config).relative_to(RUNS).as_posix()
        start = time.monotonic()
        try:
            simulate(config)
            status, error = "ok", ""
        except Exception as exc:  # log it and go on with the next run
            status, error = "failed", f"{type(exc).__name__}: {exc}"
        seconds = round(time.monotonic() - start)
        counts[status] += 1
        print(f"[{i}/{len(todo)}] {name} ... {status} ({seconds // 60}m {seconds % 60:02d}s){'  ' + error if error else ''}",
              flush=True)
        log_run({"time": datetime.now().isoformat(timespec="seconds"), "run": name, "status": status,
                 "seconds": seconds, "error": error})
        if i == len(todo) or todo[i]["sensitivity"] != config["sensitivity"]:
            forget_pairs(config["sensitivity"])
    print(f"Batch: {counts['ok']} ok, {counts['failed']} failed. Log: {RUNS / 'batch_log.csv'}")

## 4. Accuracy of every run

Collects `results/accuracy.csv` of every finished run in `runs/<hash>/` (also runs from earlier batches with the same
settings) into one table, and one summary row per run.

In [7]:
def summarize(table: pd.DataFrame) -> pd.DataFrame:
    """One row per run: legitimate sessions passed and users accepted, impostor sessions passed and users through."""
    rows = []
    for (run, n_legit, n_impostor, sens, sessions), part in table.groupby(
            ["run", "n_legitimate", "n_impostor", "sensitivity", "sessions"]):
        legit, impostor = part[part.kind == "legitimate"], part[part.kind == "impostor"]
        rows.append({"split": f"{n_legit}_{n_impostor}", "n_legitimate": n_legit, "n_impostor": n_impostor,
                     "sensitivity": sens, "sessions": sessions, "run": run,
                     "legit_passed": int(legit.passed.sum()), "legit_tested": int(legit.tested.sum()),
                     "legit_accepted": f"{int(legit.through.sum())}/{len(legit)}",
                     "impostor_passed": int(impostor.passed.sum()), "impostor_tested": int(impostor.tested.sum()),
                     "impostors_through": f"{int(impostor.through.sum())}/{len(impostor)}"})
    return pd.DataFrame(rows).sort_values(["n_legitimate", "sensitivity", "sessions"])


if SECTIONS["accuracy"]:
    group = group_folder()
    parts = [pd.read_csv(p) for p in sorted(group.glob("*/results/accuracy.csv"))]
    if not parts:
        print(f"No finished runs in {group}: run section 3 first")
    else:
        accuracy = pd.concat(parts, ignore_index=True).sort_values(
            ["n_legitimate", "n_impostor", "sensitivity", "sessions"], kind="stable")
        write_csv(accuracy, group / "accuracy.csv")
        write_csv(summarize(accuracy), group / "summary" / "summary.csv")
        print(f"{len(parts):,} runs, {len(accuracy):,} user rows")
        print(f"Output: {group / 'accuracy.csv'}, {group / 'summary' / 'summary.csv'}")

No finished runs in /home/ejang/Desktop/Thesis/thesis-mouse-biometrics/experiment_earl/runs/540c32: run section 3 first


## 5. Best sensitivity and sessions per split

Every (split, sensitivity, sessions) cell scored as (legitimate accepted + impostors rejected) / 2, so 0.5 = chance:
**user accuracy** counts users through and ranks the cells; **session accuracy** counts sessions and breaks ties
(then fewer sessions, then lower sensitivity). Chosen on the same runs it scores, so the best value is optimistic.

In [8]:
CELL = ["n_legitimate", "n_impostor", "sensitivity", "sessions"]
SPLIT = ["n_legitimate", "n_impostor"]


def score_cells(table: pd.DataFrame) -> pd.DataFrame:
    sums = table.groupby(CELL + ["kind"]).agg(passed=("passed", "sum"), tested=("tested", "sum"),
                                              through=("through", "sum"), users=("user", "size")).unstack("kind")
    cells = pd.DataFrame({
        "legit_accepted": sums["passed"]["legitimate"] / sums["tested"]["legitimate"],
        "impostor_accepted": sums["passed"]["impostor"] / sums["tested"]["impostor"],
        "legit_through": sums["through"]["legitimate"], "legit_users": sums["users"]["legitimate"],
        "impostor_through": sums["through"]["impostor"], "impostor_users": sums["users"]["impostor"],
    })
    cells["user_accuracy"] = (cells["legit_through"] / cells["legit_users"]
                              + 1 - cells["impostor_through"] / cells["impostor_users"]) / 2
    cells["session_accuracy"] = (cells["legit_accepted"] + 1 - cells["impostor_accepted"]) / 2
    return cells.reset_index()


def pick_best(cells: pd.DataFrame) -> pd.DataFrame:
    ranked = cells.sort_values(SPLIT + ["user_accuracy", "session_accuracy", "sessions", "sensitivity"],
                               ascending=[True, True, False, False, True, True])
    best = ranked.groupby(SPLIT, as_index=False).head(1).reset_index(drop=True)
    best["beats_chance"] = best["user_accuracy"] > 0.5
    return best


if SECTIONS["best"]:
    group = group_folder()
    if not (group / "accuracy.csv").exists():
        print(f"No {group / 'accuracy.csv'}: run section 4 first")
    else:
        table = pd.read_csv(group / "accuracy.csv")
        cells = score_cells(table)
        best = pick_best(cells)
        out = group / "best"
        if out.exists():
            shutil.rmtree(out)
        write_csv(cells, out / "scores.csv")
        write_csv(best, out / "best.csv")
        write_csv(best[CELL].merge(table, on=CELL), out / "best_rows.csv")
        print(best[SPLIT + ["sensitivity", "sessions", "user_accuracy", "session_accuracy", "beats_chance"]]
              .round(3).to_string(index=False))
        print(f"Output: {out}/ (scores.csv, best.csv, best_rows.csv)")

No /home/ejang/Desktop/Thesis/thesis-mouse-biometrics/experiment_earl/runs/540c32/accuracy.csv: run section 4 first


## 6. Sensitivity sweep

For each fixed split of `SWEEP` (only the users are drawn from `seed`), every library size 1 … `sessions_max` and
every sensitivity: each test session is accepted or rejected, giving TP / FP / FN / TN with positive = impostor.
A legitimate user's last `test_sessions` sessions are tested against their own library; an impostor's first
`test_sessions` against each legitimate library as a separate attempt. DTW distances are computed once per session
pair and cached (`cache/distances.pkl`); each sensitivity is a cut on them. Finished splits are kept in `parts/` and
reused while `SWEEP` and `SETTINGS` stay the same.

In [9]:
@njit(cache=True)
def _dtw_cost(a, b, band, cap, prev, cur):
    """Banded DTW cost between paths a and b, or inf once every cell exceeds `cap`."""
    n = a.shape[0]
    prev[:] = np.inf
    prev[0] = 0.0
    for i in range(n):
        cur[:] = np.inf
        row_min = np.inf
        for j in range(max(0, i - band), min(n - 1, i + band) + 1):
            dx, dy = a[i, 0] - b[j, 0], a[i, 1] - b[j, 1]
            value = np.sqrt(dx * dx + dy * dy) + min(prev[j], prev[j + 1], cur[j])
            cur[j + 1] = value
            row_min = min(row_min, value)
        if row_min > cap:
            return np.inf
        prev, cur = cur, prev
    return prev[n]


@njit(parallel=True, cache=True)
def best_distance(queries, library, band, cap):
    """Per query path: smallest DTW cost to any library path (inf if none is within `cap`)."""
    n = queries.shape[1]
    best = np.full(queries.shape[0], np.inf)
    for q in prange(queries.shape[0]):
        prev, cur = np.empty(n + 1), np.empty(n + 1)
        for k in range(library.shape[0]):
            cost = _dtw_cost(queries[q], library[k], band, min(cap, best[q]), prev, cur)
            if cost < best[q]:
                best[q] = cost
    return best


DIST_PATH = CACHE_DIR / "distances.pkl"
_DIST: dict = {}  # (query session, library session, segmentation, band, cap) -> {kind: best distance per query stroke}
if DIST_PATH.exists():
    _DIST.update(pickle.loads(DIST_PATH.read_bytes()))


def save_distances() -> None:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    tmp = DIST_PATH.with_suffix(".tmp")
    with tmp.open("wb") as handle:
        pickle.dump(_DIST, handle)
    tmp.replace(DIST_PATH)


def pair_distance(query: Session, library: Session, config: dict, cap: float) -> dict[int, np.ndarray]:
    """Best per-point DTW distance of every query stroke to the library session's strokes of the same kind."""
    n = config["resample_n"]
    key = (query.key, library.key, segmentation(config), config["dtw_band"], cap)
    if key not in _DIST:
        band = int(round(config["dtw_band"] * n))
        _DIST[key] = {kind: best_distance(paths, library.paths[kind], band, cap * n) / n
                      if len(paths) and len(library.paths[kind]) else np.full(len(paths), np.inf)
                      for kind, paths in query.paths.items()}
    return _DIST[key]


def session_curve(query: Session, library: list[Session], config: dict, cap: float) -> np.ndarray:
    """(strokes, k) best distance of each query stroke to the first 1..k library sessions (running minimum)."""
    columns = [np.concatenate([pair_distance(query, lib, config, cap)[kind] for kind in query.paths]) for lib in library]
    return np.minimum.accumulate(np.stack(columns, axis=1), axis=1)


def accepted(curve: np.ndarray, n_chunks: int, sensitivities: np.ndarray, ratio: float) -> np.ndarray:
    """(k, sensitivity) bool: does the session pass a k-session library at each sensitivity?"""
    matched = (curve[:, :, None] <= sensitivities[None, None, :]).sum(axis=0)
    return matched / max(n_chunks, 1) >= ratio


def sweep_split(legit_n: int, impostor_n: int, sens: np.ndarray) -> tuple[pd.DataFrame, pd.DataFrame]:
    """One row per (sessions, sensitivity) with TP/FP/FN/TN over test sessions and over users."""
    config = {**SETTINGS, "n_legitimate": legit_n, "n_impostor": impostor_n}
    legit, impostor, order = draw_users(config, config["seed"])
    m, t = SWEEP["sessions_max"], SWEEP["test_sessions"]
    cap = float(sens.max())
    ratio, through = config["match_shape_ratio"], config["match_session_ratio"]
    libraries = {u: [dataset_session(u, s, config) for s in order[u][:m]] for u in legit}

    def verdicts(user: str, tests: list[str], profiles: list[str]) -> np.ndarray:
        """(sessions tested x profiles, k, sensitivity): one attempt per (session, profile) pair."""
        out = []
        for name in tests:
            query = dataset_session(user, name, config)
            out.extend(accepted(session_curve(query, libraries[p], config, cap), query.n_chunks, sens, ratio)
                       for p in profiles)
        return np.array(out)

    legit_acc = {u: verdicts(u, order[u][-t:], [u]) for u in legit}
    impostor_acc = {u: verdicts(u, order[u][:t], legit) for u in impostor}
    attempts = impostor_n * t * legit_n  # impostor (session, profile) pairs
    tn = sum(a.sum(axis=0) for a in legit_acc.values())
    fn = sum(a.sum(axis=0) for a in impostor_acc.values())
    users_tn = sum(a.mean(axis=0) >= through for a in legit_acc.values())
    users_fn = sum(a.mean(axis=0) >= through for a in impostor_acc.values())

    k, s = np.meshgrid(np.arange(1, m + 1), sens, indexing="ij")
    table = pd.DataFrame({"n_legitimate": legit_n, "n_impostor": impostor_n, "sessions": k.ravel(), "sensitivity": s.ravel(),
                          "TP": (attempts - fn).ravel(), "FP": (legit_n * t - tn).ravel(),
                          "FN": fn.ravel(), "TN": tn.ravel(),
                          "users_TP": (impostor_n - users_fn).ravel(), "users_FP": (legit_n - users_tn).ravel(),
                          "users_FN": users_fn.ravel(), "users_TN": users_tn.ravel()})
    table["accuracy"] = (table["TP"] + table["TN"]) / (attempts + legit_n * t)
    table["FAR"] = table["FN"] / attempts            # impostor attempts let in
    table["FRR"] = table["FP"] / (legit_n * t)      # legitimate sessions rejected
    table["balanced_accuracy"] = 1 - (table["FAR"] + table["FRR"]) / 2
    table["users_accuracy"] = (table["users_TP"] + table["users_TN"]) / (legit_n + impostor_n)
    table["legitimate"] = ",".join(legit)
    table["impostor"] = ",".join(impostor)
    return table, user_table(legit_n, impostor_n, legit, legit_acc, impostor_acc, k, s)


def user_table(legit_n: int, impostor_n: int, legit: list[str], legit_acc: dict, impostor_acc: dict,
               k: np.ndarray, s: np.ndarray) -> pd.DataFrame:
    """One confusion matrix per (legitimate user, half, sessions, sensitivity): the user's own test sessions and every
    impostor attempt against the user's library. Test sessions alternate between halves (even = pick, odd = check),
    so section 7 can choose a setting on one half and score it on sessions it never saw."""
    rows = []
    for p, u in enumerate(legit):
        genuine = legit_acc[u]
        attacks = np.concatenate([a[p::legit_n] for a in impostor_acc.values()])  # attempts are (session, profile)
        for half, start in (("pick", 0), ("check", 1)):
            g, a = genuine[start::2], attacks[start::2]
            tn, fn = g.sum(axis=0), a.sum(axis=0)
            rows.append(pd.DataFrame({"n_legitimate": legit_n, "n_impostor": impostor_n, "user": u, "half": half,
                                      "sessions": k.ravel(), "sensitivity": s.ravel(),
                                      "TP": (len(a) - fn).ravel(), "FP": (len(g) - tn).ravel(),
                                      "FN": fn.ravel(), "TN": tn.ravel()}))
    table = pd.concat(rows, ignore_index=True)
    table["FAR"] = table["FN"] / (table["TP"] + table["FN"])
    table["FRR"] = table["FP"] / (table["FP"] + table["TN"])
    table["balanced_accuracy"] = 1 - (table["FAR"] + table["FRR"]) / 2
    return table


if SECTIONS["sweep"]:
    start, stop, step = SWEEP["sensitivity"]
    sens = np.round(np.arange(start, stop + step / 2, step), 4)
    folder = group_folder() / "sweep"
    plan_text = yaml.safe_dump({**SETTINGS, "sweep": SWEEP}, sort_keys=False)
    if (folder / "config.yaml").exists() and (folder / "config.yaml").read_text() != plan_text:
        shutil.rmtree(folder / "parts", ignore_errors=True)  # SWEEP changed: earlier splits no longer apply
    (folder / "parts").mkdir(parents=True, exist_ok=True)
    (folder / "config.yaml").write_text(plan_text)
    parts, users = [], []
    for legit_n, impostor_n in SWEEP["splits"]:
        path = folder / "parts" / f"{legit_n}_{impostor_n}.csv"
        user_path = folder / "parts" / f"{legit_n}_{impostor_n}_users.csv"
        if path.exists() and user_path.exists():
            parts.append(pd.read_csv(path))
            users.append(pd.read_csv(user_path))
            print(f"[{legit_n} L, {impostor_n} I] finished earlier, reused")
            continue
        start_time = time.monotonic()
        part, user_part = sweep_split(legit_n, impostor_n, sens)
        save_distances()
        write_csv(part, path)
        write_csv(user_part, user_path)
        parts.append(part)
        users.append(user_part)
        print(f"[{legit_n} L, {impostor_n} I] done ({time.monotonic() - start_time:.0f} s)", flush=True)
    write_csv(pd.concat(parts), folder / "confusion.csv")
    write_csv(pd.concat(users), folder / "user_confusion.csv")
    print(f"{len(SWEEP['splits'])} splits x {SWEEP['sessions_max']} sessions x {len(sens)} sensitivities "
          f"= {sum(len(p) for p in parts):,} confusion matrices")
    print(f"Output: {folder}/ (confusion.csv, user_confusion.csv, parts/, config.yaml)")

[1 L, 5 I] done (93 s)
[5 L, 1 I] done (98 s)
[3 L, 2 I] done (75 s)
[2 L, 3 I] done (30 s)
4 splits x 40 sessions x 100 sensitivities = 16,000 confusion matrices
Output: /home/ejang/Desktop/Thesis/thesis-mouse-biometrics/experiment_earl/runs/540c32/sweep/ (confusion.csv, user_confusion.csv, parts/, config.yaml)


## 7. Best setting per legitimate user

From section 6: per legitimate user and per split, the setting (sensitivity, library size) with the highest balanced
accuracy on the **pick** half of the test sessions (ties: fewer sessions, then lower sensitivity), scored unchanged on
the **check** half. Report the check-half score; the pick-half score is the best of many settings and optimistic.

In [10]:
SETTING = ["sessions", "sensitivity"]
COUNTS = ["TP", "FP", "FN", "TN"]


def rates(table: pd.DataFrame) -> pd.DataFrame:
    table = table.copy()
    table["FAR"] = table["FN"] / (table["TP"] + table["FN"])
    table["FRR"] = table["FP"] / (table["FP"] + table["TN"])
    table["balanced_accuracy"] = 1 - (table["FAR"] + table["FRR"]) / 2
    return table


def pick(table: pd.DataFrame, keys: list[str]) -> pd.DataFrame:
    """Per group: the best pick-half setting, with its check-half scores attached unchanged."""
    halves = {h: part.drop(columns="half") for h, part in table.groupby("half")}
    ranked = halves["pick"].assign(score=halves["pick"]["balanced_accuracy"].round(9)).sort_values(
        keys + ["score", "sessions", "sensitivity"], ascending=[True] * len(keys) + [False, True, True])
    best = ranked.groupby(keys, sort=False).head(1).drop(columns="score")
    check = halves["check"][keys + SETTING + COUNTS + ["FAR", "FRR", "balanced_accuracy"]]
    return best.merge(check, on=keys + SETTING, suffixes=("_pick", "_check")).reset_index(drop=True)


if SECTIONS["best_setting"]:
    folder = group_folder() / "sweep"
    if not (folder / "user_confusion.csv").exists():
        print(f"No {folder / 'user_confusion.csv'}: run section 6 first")
    else:
        table = pd.read_csv(folder / "user_confusion.csv")
        best_users = pick(table, SPLIT + ["user"])
        best_splits = pick(rates(table.groupby(SPLIT + ["half"] + SETTING, as_index=False)[COUNTS].sum()), SPLIT)
        write_csv(best_users, folder / "best" / "best_users.csv")
        write_csv(best_splits, folder / "best" / "best_splits.csv")
        columns = ["sensitivity", "sessions", "balanced_accuracy_pick", "balanced_accuracy_check", "FAR_check", "FRR_check"]
        print("Per legitimate user:")
        print(best_users[SPLIT + ["user"] + columns].round(3).to_string(index=False))
        print("\nPer split:")
        print(best_splits[SPLIT + columns].round(3).to_string(index=False))
        print(f"Output: {folder / 'best'}/ (best_users.csv, best_splits.csv)")

Per legitimate user:
 n_legitimate  n_impostor   user  sensitivity  sessions  balanced_accuracy_pick  balanced_accuracy_check  FAR_check  FRR_check
            1           5  user9         0.02        17                   0.675                    0.635      0.380       0.35
            2           3 user20         0.07         2                   0.717                    0.575      0.550       0.30
            2           3  user9         0.02        21                   0.683                    0.658      0.533       0.15
            3           2 user20         0.07         2                   0.775                    0.600      0.500       0.30
            3           2 user23         0.04         6                   0.600                    0.512      0.675       0.30
            3           2  user9         0.03         4                   0.688                    0.688      0.375       0.25
            5           1 user12         0.07         3                   0.600           

## 8. Matched stroke pairs (optional)

For `SINGLE_RUN`: per legitimate user and session step k, every impostor stroke (first k sessions) that matches a
stroke in the user's first k sessions, with the first matching stroke and its DTW cost per point. The printed verdict
per impostor uses the same rules as the batch.

In [11]:
@njit(parallel=True, cache=True)
def first_match(queries, library, band, cap):
    """Per query path: index of the first library path within DTW cost `cap`, or -1 (same search as has_match)."""
    n = queries.shape[1]
    found = np.full(queries.shape[0], -1, dtype=np.int64)
    for q in prange(queries.shape[0]):
        prev, cur = np.empty(n + 1), np.empty(n + 1)
        for k in range(library.shape[0]):
            if _dtw_within(queries[q], library[k], band, cap, prev, cur):
                found[q] = k
                break
    return found


@njit(cache=True)
def dtw_cost(a, b, band):
    """Full banded DTW cost between paths a and b."""
    n = a.shape[0]
    cost = np.full((n + 1, n + 1), np.inf)
    cost[0, 0] = 0.0
    for i in range(n):
        for j in range(max(0, i - band), min(n - 1, i + band) + 1):
            d = np.sqrt((a[i, 0] - b[j, 0]) ** 2 + (a[i, 1] - b[j, 1]) ** 2)
            cost[i + 1, j + 1] = d + min(cost[i, j], cost[i, j + 1], cost[i + 1, j])
    return cost[n, n]


def step_matches(profile: str, impostor: list[str], order: dict, k: int, config: dict, pairs: dict) -> list[dict]:
    """Every impostor stroke (first k sessions) with a matching stroke in the profile's first k sessions."""
    n = config["resample_n"]
    band, cap = int(round(config["dtw_band"] * n)), config["sensitivity"] * n
    library = [dataset_session(profile, name, config) for name in order[profile][:k]]
    rows = []
    for user in impostor:
        for name in order[user][:k]:
            query = dataset_session(user, name, config)
            for kind, paths in query.paths.items():
                best = np.full(len(paths), -1)
                source = np.full(len(paths), -1)
                for j, lib in enumerate(library):
                    key = (query.key, lib.key, kind)
                    if key not in pairs:
                        pairs[key] = first_match(paths, lib.paths[kind], band, cap) if len(lib.paths[kind]) and len(paths) \
                            else np.full(len(paths), -1)
                    new = (best < 0) & (pairs[key] >= 0)
                    best[new], source[new] = pairs[key][new], j
                for q in np.flatnonzero(best >= 0):
                    lib = library[source[q]]
                    rows.append({"profile": profile, "sessions": k, "impostor": user, "impostor_session": name,
                                 "kind": KIND_NAMES[kind], "chunk": int(q), "profile_session": lib.key.split("/")[-1],
                                 "profile_chunk": int(best[q]),
                                 "cost_per_point": dtw_cost(paths[q], lib.paths[kind][best[q]], band) / n})
    return rows


def session_verdicts(rows: list[dict], impostor: list[str], order: dict, k: int, config: dict) -> list[str]:
    """Per impostor: matched strokes and sessions at the batch's matching rules."""
    lines = []
    for user in impostor:
        ratios, strokes, matched = [], 0, 0
        for name in order[user][:k]:
            session = dataset_session(user, name, config)
            hits = sum(r["impostor"] == user and r["impostor_session"] == name for r in rows)
            ratios.append(hits / max(session.n_chunks, 1))
            strokes += session.n_chunks
            matched += hits
        hit_sessions = sum(r >= config["match_shape_ratio"] for r in ratios)
        verdict = "MATCHED" if hit_sessions / len(ratios) >= config["match_session_ratio"] else "not matched"
        lines.append(f"{user}: {matched:,}/{strokes:,} strokes match, {hit_sessions}/{len(ratios)} sessions "
                     f"≥ {config['match_shape_ratio']:.0%} → {verdict}")
    return lines


if SECTIONS["matches"]:
    config = {**SETTINGS, **SINGLE_RUN}
    folder = run_folder(config)
    if not (folder / "config.yaml").exists():
        create_run(folder, config)
    legit, impostor, order = draw_users(config, config["seed"])
    pairs, table = {}, []
    for profile in legit:
        for k in range(1, config["sessions_per_user"] + 1):
            rows = step_matches(profile, impostor, order, k, config, pairs)
            table += rows
            print(f"{profile} {k} sessions: {len(rows):,} matched strokes | "
                  + " | ".join(session_verdicts(rows, impostor, order, k, config)), flush=True)
    write_csv(pd.DataFrame(table).round(4), folder / "results" / "matches.csv")
    print(f"Output: {folder / 'results' / 'matches.csv'}")

## 9. Genuine check (optional)

For `SINGLE_RUN`: per legitimate user and step k, the user's next `GENUINE_HELD_OUT` sessions after
`sessions_per_user` (never in the profile) and the impostors' first k sessions, against the user's first-k-session
library. A useful profile accepts its owner more often than it accepts impostors.

In [12]:
def genuine_score(user: str, sessions: list[str], library: list[Session], kind: str, profile: str, k: int,
                  config: dict) -> list[dict]:
    """Share of each session's strokes that match the library, and whether it reaches match_shape_ratio."""
    rows = []
    for name in sessions:
        session = dataset_session(user, name, config)
        ratio = matched_chunks(session, library, config) / max(session.n_chunks, 1)
        rows.append({"profile": profile, "sessions": k, "kind": kind, "user": user, "session": name,
                     "strokes": session.n_chunks, "ratio": round(ratio, 4),
                     "matched": ratio >= config["match_shape_ratio"]})
    return rows


if SECTIONS["genuine_check"]:
    config = {**SETTINGS, **SINGLE_RUN}
    folder = run_folder(config)
    if not (folder / "config.yaml").exists():
        create_run(folder, config)
    legit, impostor, order = draw_users(config, config["seed"])
    steps = list(range(1, config["sessions_per_user"] + 1))
    held_out = {u: order[u][config["sessions_per_user"]:][:GENUINE_HELD_OUT] for u in legit}
    rows = []
    for profile in legit:
        for k in steps:
            library = [dataset_session(profile, n, config) for n in order[profile][:k]]
            rows += genuine_score(profile, held_out[profile], library, "genuine", profile, k, config)
            for user in impostor:
                rows += genuine_score(user, order[user][:k], library, "impostor", profile, k, config)
        print(f"{profile} done", flush=True)
    save_pairs()
    table = pd.DataFrame(rows)
    summary = (table.groupby(["profile", "sessions", "kind"], sort=False)
               .agg(mean_ratio=("ratio", "mean"), passed=("matched", "sum"), tested=("matched", "size")).reset_index())
    write_csv(table, folder / "results" / "genuine_check.csv")
    write_csv(summary.round(4), folder / "results" / "genuine_check_summary.csv")
    print(f"Output: {folder / 'results'}/ (genuine_check.csv, genuine_check_summary.csv)")